# SPY 0DTE Breakout + Reversal Strategy

**Setup:** Opening range = first 30 min. Two parallel scans, first to fire wins. Single trade/day. No Fridays.

**Breakout (call/put):**
- 2 consecutive 1-min closes outside ORH/ORL
- Price aligned with VWAP, EMA9, EMA21
- 5-min cross-day RSI > 50 (call) / < 50 (put)
- Skip if RSI was > 70 or < 30 in the previous 5-min bar

**Reversal (call/put):**
- 1-min candle closes outside OR, next 1-min candle closes back inside
- Direction by which side broke: dip below ORL = CALL; spike above ORH = PUT
- VWAP + EMA9 aligned (no EMA21)
- BOTH cross-day AND intraday 5-min RSI > 50 (call) / < 50 (put)
- Skip call reversals if cross-day RSI in [60, 65]

**Exits:** 10% net profit target, 20% net stop loss, time stop (sweepable 30/60/120 min).

## 1. Setup

In [ ]:
import os, sys
REPO = '/content/iron-condor'
BRANCH = 'claude/spy-options-trading-bot-ri4Ms'
if not os.path.isdir(REPO):
    !git clone --quiet -b {BRANCH} https://github.com/coolin815/iron-condor.git {REPO}
else:
    !cd {REPO} && git pull --quiet
SRC = REPO + '/src'
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.chdir(REPO)
print('OK')

## 2. Paste your Polygon key

In [ ]:
import os, getpass
os.environ['POLYGON_API_KEY'] = getpass.getpass('Polygon API key: ')
print('Key loaded:', len(os.environ['POLYGON_API_KEY']), 'chars')

## 3. Sweep the new strategy over the cached year

Sweep:
- 3 signal modes: both / breakout-only / reversal-only
- 3 time stops: 30 / 60 / 120 min
- 3 entry cutoffs: 11:30 / 13:00 / 15:30 ET
= **27 configs** over 1 year. Cache is warm so ~1 minute.

In [ ]:
!python -m iron_condor.cli --days 365 --sweep

## 4. Top configs

In [ ]:
import pandas as pd
summary = pd.read_csv('results/sweep_summary.csv')
summary.head(20)

## 5. Performance by signal mode and direction

In [ ]:
trades = pd.read_csv('results/sweep_trades.csv')
taken = trades[trades['exit_reason'].isin(['profit', 'stop', 'time_stop'])]

print('=== By signal type (breakout vs reversal) ===')
print(taken.groupby('signal_type').agg(
    trades=('net_pnl', 'count'),
    win_rate=('net_pnl', lambda s: (s > 0).mean()),
    avg_pnl=('net_pnl', 'mean'),
    median_pnl=('net_pnl', 'median'),
    total_pnl=('net_pnl', 'sum'),
).round(2))

print('\n=== By direction (call vs put) ===')
print(taken.groupby('signal_direction').agg(
    trades=('net_pnl', 'count'),
    win_rate=('net_pnl', lambda s: (s > 0).mean()),
    avg_pnl=('net_pnl', 'mean'),
    median_pnl=('net_pnl', 'median'),
    total_pnl=('net_pnl', 'sum'),
).round(2))

## 6. Timing — how long do winners take?

In [ ]:
from iron_condor.metrics import analyze_timing
print(analyze_timing(trades))

## 7. Equity curves — top 3 configs

In [ ]:
import matplotlib.pyplot as plt

top3 = summary.head(3)['config'].tolist()
fig, ax = plt.subplots(figsize=(10, 5))
for cfg in top3:
    sub = trades[trades['config'] == cfg].sort_values('day')
    ax.plot(pd.to_datetime(sub['day']), sub['balance_after'], label=cfg, linewidth=1.5)
ax.axhline(1500, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Date'); ax.set_ylabel('Account balance ($)')
ax.set_title('Top-3 config equity curves')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()